# NDBC buoys (tabledap -> DataFrame)

A `tabledap` dataset returns record tables as a `pandas.DataFrame`. Here we pull a day of NDBC standard-meteorological buoy water-temperature observations off the U.S. west coast.

## Setup + request

`OUTPUT_KIND` is `tabular`, so `download()` returns a frame (also written to CSV). `aggregate=` would be rejected for a table.

In [ ]:
import tempfile
from pathlib import Path

from earthlens.core import EarthLens

out_dir = Path(tempfile.mkdtemp(prefix='earthlens-erddap-ndbc-'))
df = EarthLens(
    data_source='erddap',
    dataset='cwwcNDBCMet',
    variables=['station', 'time', 'WTMP'],
    start='2023-01-01',
    end='2023-01-01',
    lat_lim=[36.0, 38.0],
    lon_lim=[-124.0, -122.0],
    path=out_dir,
).download()

print('shape  :', df.shape)
print('columns:', list(df.columns))

## Peek at the records

ERDDAP appends units to the column names (e.g. `WTMP (degree_C)`).

In [ ]:
df.head()

## Per-station water temperature

In [ ]:
wtmp_col = [c for c in df.columns if c.startswith('WTMP')][0]
summary = (
    df.dropna(subset=[wtmp_col])
    .groupby('station')[wtmp_col]
    .agg(['count', 'mean', 'min', 'max'])
    .round(2)
)
print(f'{len(summary)} station(s) reported water temperature:')
summary